In [120]:
import pandas as pd
import numpy as np

In [121]:
# Reading mnist_784.csv dataset.
df = pd.read_csv('mnist_784.csv')

In [122]:
# Hyper parameters.
neural_network = [784, 128, 128, 128, 128, 128, 10]
input_size = neural_network[0]
learning_rate = 0.005
epochs = 40
batch_size = 100
training_set_percentage = 0.70
test_set_percentage = 1 - training_set_percentage
np.random.seed(42)

In [123]:
# Inputs.
X = df.iloc[:,0:784].values/255
# Outputs.
Y0 = df.iloc[:,784].values

# One hot-encoding.
Y=np.zeros((70000,10)) 
for i in range(70000): 
    Y[i,Y0[i]]=1

# Training sets. 
training_size = int(len(X) * training_set_percentage)
X_training_set = X[:training_size, :].T
Y_training_set = Y[:training_size, :].T

# Test sets.
X_test_set = X[training_size:, :].T
Y_test_set = Y[training_size:, :].T

In [124]:
# Initialize neural network weights and biases (model parameters).
weights = []
biases = []

# Initialize weights with small random values.
for current_in, next_out in zip(neural_network, neural_network[1:]):
    weights.append(np.random.randn(next_out, current_in))

# Initialize biases with zeros.
for layer_size in neural_network[1:]:
    biases.append(np.zeros((layer_size, 1)))

In [125]:
# Training.
for epoch in range(epochs):

    error = 0
    num_batches = X_training_set.shape[1] // batch_size
    
    for batch in range(num_batches):
        start = batch * batch_size
        end = (batch+1) * batch_size
        X_batch_training_set=X_training_set[:,start:end]
        Y_batch_training_set=Y_training_set[:,start:end]

        # Forward pass over the whole batch at once.

        # Initialize input matrix: each column is one image.
        input_matrix = X_batch_training_set

        # Initialize label matrix: each column is one one-hot label.
        actual_label = Y_batch_training_set
        
        # Forward propagation.
        # Start with the input_matrix.
        activated_neuron = input_matrix
        activated_neurons = []
        activated_neurons.append(activated_neuron)

        # Iterate through the layers.
        for layer in range(len(neural_network)-1):
            activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))
            activated_neurons.append(activated_neuron)

        # Last layer is the output.
        output_neurons = activated_neuron

        # Calculating the error using the mean absolute error loss function.
        error += np.sum(np.square(actual_label - output_neurons))
        
        # Backpropagation.
        
        # Calculate the delta.
        l_wr_o = 2 * (output_neurons - actual_label)
        o_wr_z = output_neurons * (1 - output_neurons)
        delta = l_wr_o * o_wr_z

        # Go backwards through layers.
        for layer in range(len(weights) - 1, -1, -1):

            # With respect to weight.
            z_wr_weight = activated_neurons[layer]
            # With respect to bias.
            z_wr_bias = np.ones((1, batch_size))

            # Gradients.
            l_wr_weight = (delta @ z_wr_weight.T)
            l_wr_bias = (delta @ z_wr_bias.T)

            # With respect to previous neuron.
            z_wr_q = weights[layer]
            q_wr_m = activated_neurons[layer] * (1 - activated_neurons[layer])
            # Next delta.
            next_delta = (z_wr_q.T @ delta) * q_wr_m
            
            # Gradient descent.
            weights[layer] -= learning_rate * l_wr_weight
            biases[layer] -= learning_rate * l_wr_bias

            delta = next_delta
            
    print(f"Epoch: {epoch}, Error: {error}")

Epoch: 0, Error: 64301.26456824801
Epoch: 1, Error: 25328.984492821623
Epoch: 2, Error: 19101.566262301494
Epoch: 3, Error: 16445.60800944989
Epoch: 4, Error: 14856.022156938472
Epoch: 5, Error: 13763.074364251142
Epoch: 6, Error: 12959.544365867152
Epoch: 7, Error: 12317.9100416034
Epoch: 8, Error: 11788.00595265675
Epoch: 9, Error: 11346.226160237853
Epoch: 10, Error: 10968.906177415587
Epoch: 11, Error: 10636.160892492713
Epoch: 12, Error: 10341.293646408398
Epoch: 13, Error: 10075.357022946666
Epoch: 14, Error: 9834.750485857041
Epoch: 15, Error: 9617.375566591554
Epoch: 16, Error: 9421.284735936124
Epoch: 17, Error: 9244.088583504976
Epoch: 18, Error: 9082.090251941481
Epoch: 19, Error: 8932.143210628587
Epoch: 20, Error: 8792.971936784921
Epoch: 21, Error: 8663.222398887217
Epoch: 22, Error: 8542.795424788532
Epoch: 23, Error: 8430.338195127444
Epoch: 24, Error: 8323.101192736129
Epoch: 25, Error: 8220.565728357566
Epoch: 26, Error: 8122.912944347547
Epoch: 27, Error: 8029.978490

In [126]:
correct = 0

# Starting neuron.
activated_neuron = X_test_set

# Iterate through the layers.
for layer in range(len(neural_network)-1):
    activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))

# Last layer is the output.
output_neurons = activated_neuron

# Predicted and actual digit for every test image (down each column).
predicted_digits = np.argmax(output_neurons, axis=0)
actual_digits = np.argmax(Y_test_set, axis=0)

# Count the matches.
correct = np.sum(predicted_digits == actual_digits)

# Calculate accuracy.
accuracy = correct / Y_test_set.shape[1] * 100

print("Correct predictions:", correct)
print("Total test images:", Y_test_set.shape[1])
print(f"Test accuracy: {accuracy:.2f}%")

Correct predictions: 19308
Total test images: 21000
Test accuracy: 91.94%
